In [0]:
ss = 'shoe'

In [0]:
#!/usr/bin/env python3
"""
Download WAQI COVID-19 air quality datasets.
"""
import os
import sys
import requests
from pathlib import Path

BASE_URL = "https://aqicn.org/data-platform/covid19/report/48852-12462758"
OUTDIR = "waqi-covid-data"

# Year / period list
PERIODS = [
    "2015H1",
    "2016H1",
    "2017H1",
    "2018H1",
    "2019Q1", "2019Q2", "2019Q3", "2019Q4",
    "2020Q1", "2020Q2", "2020Q3", "2020Q4",
    "2021Q1", "2021Q2", "2021Q3", "2021Q4",
    "2022Q1", "2022Q2", "2022Q3", "2022Q4",
    "2023Q1", "2023Q2", "2023Q3", "2023Q4",
    "2026",
]


def download_file(url, output_path):
    """Download a file from URL to output_path with compression support."""
    headers = {
        'Accept-Encoding': 'gzip, deflate, br',
        'User-Agent': 'Mozilla/5.0 (compatible; WAQI-Downloader/1.0)'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        with open(output_path, 'wb') as f:
            f.write(response.content)
        
        return True
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url}: {e}", file=sys.stderr)
        return False


def main():
    # Create output directory
    Path(OUTDIR).mkdir(parents=True, exist_ok=True)
    print(f"Downloading WAQI COVID-19 datasets into {OUTDIR}...")
    
    # Download period datasets
    failed = []
    for period in PERIODS:
        outfile = os.path.join(OUTDIR, f"waqi-covid-{period}.csv")
        url = f"{BASE_URL}/{period}"
        print(f"  → {outfile}")
        
        if not download_file(url, outfile):
            failed.append(period)
    
    # Download station metadata
    print("Downloading station metadata...")
    metadata_url = "https://aqicn.org/data-platform/covid19/airquality-covid19-cities.json"
    metadata_file = os.path.join(OUTDIR, "airquality-covid19-cities.json")
    
    if not download_file(metadata_url, metadata_file):
        failed.append("metadata")
    
    # Report results
    if failed:
        print(f"\nWarning: Failed to download {len(failed)} file(s): {', '.join(failed)}")
        sys.exit(1)
    else:
        print("\nAll downloads completed successfully.")


if __name__ == "__main__":
    main()